# Pipeline Gardenia | Etapa 7.6: Ecossistema Hadoop (HDFS, YARN, MapReduce, Hive, Pig, Sqoop e HBase)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Anlbp/Projeto-Gardenia-unifeob/blob/main/recursos/dados/notebook_dados_nao_filtrados/07_6_hadoop_ecossistema.ipynb)

Mostra como os dados do projeto seriam **armazenados e processados de forma distribuída** com o ecossistema Hadoop, e o que cada ferramenta faz no contexto da Gardênia.

## O que é executado aqui e o que é roteiro

Hadoop é um conjunto de serviços que precisa ser instalado e iniciado (NameNode, DataNode, ResourceManager...). Este notebook roda no Colab, que não tem isso. Por isso:

| | Executado neste notebook | Só escrito (roteiro para a VM) |
|---|---|---|
| **MapReduce** | sim: a lógica dos mappers e reducers roda numa simulação do Hadoop Streaming e o resultado é conferido com o pandas | rodar no cluster (`executar_mapreduce.sh`) |
| **HBase** | sim: geração do arquivo com a chave de linha e a leitura por prefixo (simulada) | criar a tabela e carregar (`hbase_gardenia.sh`) |
| **HDFS, YARN, Hive, Pig, Sqoop** | não | scripts prontos, sem execução |

Se o Hadoop estiver instalado na máquina onde o notebook roda, as células de HDFS e MapReduce rodam de verdade (elas testam `hdfs` e `hadoop` no PATH). **Nenhum script de cluster foi executado durante o desenvolvimento**: eles seguem a documentação das ferramentas e precisam ser testados na VM 3.

Todos os arquivos são gravados na pasta `hadoop/` e também acompanham o projeto como arquivos separados, para uso direto na VM.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Cada ferramenta no contexto do projeto

| Ferramenta | Papel no Hadoop | Uso na Gardênia | Faz sentido aqui? |
|---|---|---|---|
| **HDFS** | Sistema de arquivos distribuído. Divide cada arquivo em blocos (128 MB por padrão) e guarda cópias em várias máquinas | Guardar as zonas `raw`, `trusted` e `quarentena` em `/gardenia/...` | **Sim, em escala maior.** Com os volumes atuais (poucos MB por lote) cada arquivo cabe em um bloco só, e a distribuição não traz ganho. |
| **YARN** | Gerenciador de recursos do cluster. Reparte CPU e memória entre os jobs | Agendar os jobs de MapReduce e o job Spark da VM 3 | **Sim, se houver mais de uma máquina.** Numa só VM ele só serviria para mostrar o conceito. |
| **MapReduce** | Modelo de processamento em lote: `map` (transforma), `shuffle and sort` (agrupa por chave) e `reduce` (agrega) | Receita líquida por categoria, com junção entre compras e produtos | **Sim.** É o modelo que o Spark usa por baixo, e a junção mostra bem por que o `shuffle` existe. |
| **Hive** | SQL sobre arquivos do HDFS (compila a consulta para jobs distribuídos) | As mesmas 4 consultas do notebook 7.5, em HiveQL | **Sim.** Boa opção para quem prefere SQL a código. |
| **Pig** | Linguagem de fluxo de dados (Pig Latin) que vira jobs MapReduce | A mesma receita por categoria, em poucas linhas | **Parcial.** Hoje o Spark e o Hive cobrem o mesmo papel; Pig aparece mais em sistemas antigos. |
| **Sqoop** | Transfere dados entre banco relacional e HDFS | Importar compras de um MySQL/PostgreSQL | **Não aplicável hoje**: a fonte são arquivos CSV do simulador, copiados por SCP. Passaria a fazer sentido se as compras ficassem num banco relacional. O projeto Sqoop foi desativado pela Apache; o equivalente atual é o Spark lendo por JDBC. |
| **HBase** | Banco NoSQL colunar, com leitura e escrita rápidas por chave | Histórico de compras de um cliente em milissegundos | **Parcial.** Serve para consulta pontual (atendimento), não para agregar o ano inteiro; isso fica com Spark e Hive. |

**Como as peças se encaixam:** os CSVs limpos (`trusted`) vão para o **HDFS**; o **YARN** distribui o trabalho; o **MapReduce**, o **Hive**, o **Pig** e o **Spark** leem do HDFS e calculam os resultados; o **HBase** guarda o que precisa de acesso individual rápido; o **Sqoop** seria a porta de entrada se a origem fosse um banco relacional.

## 2. Configuração e leitura dos dados

In [3]:
# ============================ CONFIGURAÇÃO ============================
FONTE = "drive"             # "drive" | "github" | "upload" | "local"
LOTES = None                # None = todos os lotes encontrados. Ou uma lista, ex.: [1, 2]

DRIVE_PASTA = "/content/drive/MyDrive/gardenia"   # o 03 grava os trusted em {DRIVE_PASTA}/saida
PASTA_TRUSTED = f"{DRIVE_PASTA}/saida"

# --- usado quando FONTE == "github" (os trusted versionados no repositório) ---
GITHUB_USUARIO = "Anlbp"
GITHUB_REPO    = "Projeto-Gardenia-unifeob"
GITHUB_BRANCH  = "main"
PASTA_GITHUB   = "recursos/dados/trusted"
LOTES_GITHUB   = [1]        # quais lotes baixar do GitHub

# --- usado quando FONTE == "local" ---
PASTA_LOCAL = "dados_trusted"
# =======================================================================

In [4]:
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

import urllib.error
import urllib.request

try:
    if FONTE == "drive":
        from google.colab import drive
        drive.mount("/content/drive")
        PASTA_DADOS = PASTA_TRUSTED

    elif FONTE == "github":
        PASTA_DADOS = "dados_trusted"
        os.makedirs(PASTA_DADOS, exist_ok=True)
        branch = GITHUB_BRANCH.removeprefix("tree/")
        for n in LOTES_GITHUB:
            for t in ["clientes", "produtos", "compras"]:
                nome = f"trusted_{t}_{n}.csv"
                url = f"https://raw.githubusercontent.com/{GITHUB_USUARIO}/{GITHUB_REPO}/{branch}/{PASTA_GITHUB}/{nome}"
                print("baixando", url)
                urllib.request.urlretrieve(url, os.path.join(PASTA_DADOS, nome))

    elif FONTE == "upload":
        from google.colab import files
        PASTA_DADOS = "dados_trusted"
        os.makedirs(PASTA_DADOS, exist_ok=True)
        print("Selecione os 3 arquivos trusted_*_N.csv de cada lote (gerados pelo notebook 03)")
        for nome, conteudo in files.upload().items():
            with open(os.path.join(PASTA_DADOS, nome), "wb") as f:
                f.write(conteudo)

    elif FONTE == "local":
        PASTA_DADOS = PASTA_LOCAL

    else:
        raise ValueError('FONTE deve ser "drive", "github", "upload" ou "local".')
except urllib.error.HTTPError as e:
    raise FileNotFoundError(f"Não consegui baixar o arquivo ({e.code}). Confira usuário, repositório, pasta e lotes.") from None

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
# separa só os lotes pedidos numa pasta de trabalho (o MapReduce lê uma pasta de arquivos)
ENTRADA = "dados_trusted_mr"
shutil.rmtree(ENTRADA, ignore_errors=True)
os.makedirs(ENTRADA)
lotes = sorted({int(m.group(1)) for n in os.listdir(PASTA_DADOS)
                if (m := re.fullmatch(r"trusted_clientes_(\d+)\.csv", n))
                and all(os.path.exists(os.path.join(PASTA_DADOS, f"trusted_{t}_{m.group(1)}.csv"))
                        for t in ["clientes", "produtos", "compras"])})
if LOTES is not None:
    lotes = [n for n in lotes if n in LOTES]
if not lotes:
    raise FileNotFoundError(f"Não encontrei lotes completos em {PASTA_DADOS}. Rode antes o notebook 03 e confira FONTE e as pastas.")
for n in lotes:
    for t in ["clientes", "produtos", "compras"]:
        shutil.copy(os.path.join(PASTA_DADOS, f"trusted_{t}_{n}.csv"), ENTRADA)
print("Lotes:", lotes)

os.makedirs("hadoop", exist_ok=True)


def gravar(nome, conteudo):
    caminho = Path("hadoop") / nome
    caminho.write_text(conteudo, encoding="utf-8")
    if nome.endswith((".sh", ".py")):
        caminho.chmod(0o755)
    print(f"gravado: {caminho}  ({len(conteudo.splitlines())} linhas)")


def tem(comando):
    return shutil.which(comando) is not None


print("Hadoop instalado nesta máquina:", "sim" if tem("hadoop") and tem("hdfs") else "não (as células de cluster só mostram o roteiro)")

Lotes: [1]
Hadoop instalado nesta máquina: não (as células de cluster só mostram o roteiro)


## 3. HDFS

O HDFS tem dois tipos de serviço: o **NameNode** guarda o mapa (qual arquivo está em quais blocos e em quais máquinas) e os **DataNodes** guardam os blocos. Cada bloco tem cópias em máquinas diferentes (replicação, 3 por padrão), então a queda de uma máquina não perde dado. O processamento vai até onde o bloco está (em vez de levar o dado até o código), e é isso que permite ler arquivos grandes em paralelo.

**Layout escolhido para o projeto:** uma pasta por zona e uma subpasta por tabela, porque Hive e MapReduce leem **pastas**.

```
/gardenia/raw/{clientes,produtos,compras}          CSV como chegou da VM 1
/gardenia/trusted/{clientes,produtos,compras}      CSV limpo (saída da VM 2)
/gardenia/quarentena/{clientes,produtos,compras}   registros rejeitados, com o motivo
/gardenia/mr   /gardenia/pig   /gardenia/hbase   /gardenia/warehouse    resultados e áreas de trabalho
```

In [6]:
# quanto os dados ocupam, em comparação com o tamanho de um bloco do HDFS (128 MB por padrão)
BLOCO_MB = 128
total_mb = sum(os.path.getsize(os.path.join(ENTRADA, n)) for n in os.listdir(ENTRADA)) / 1e6
maior = max(os.listdir(ENTRADA), key=lambda n: os.path.getsize(os.path.join(ENTRADA, n)))
print(f"Dados lidos: {total_mb:.2f} MB no total; maior arquivo: {maior} ({os.path.getsize(os.path.join(ENTRADA, maior)) / 1e6:.2f} MB)")
print(f"Um bloco do HDFS tem {BLOCO_MB} MB: cada arquivo ocupa 1 bloco (menos de {100 * os.path.getsize(os.path.join(ENTRADA, maior)) / 1e6 / BLOCO_MB:.1f}% dele).")
print("Conclusão: com este volume o HDFS não acelera nada. Ele passa a valer quando os arquivos têm centenas de MB ou mais, ou quando há muitos lotes.")

Dados lidos: 0.79 MB no total; maior arquivo: trusted_clientes_1.csv (0.54 MB)
Um bloco do HDFS tem 128 MB: cada arquivo ocupa 1 bloco (menos de 0.4% dele).
Conclusão: com este volume o HDFS não acelera nada. Ele passa a valer quando os arquivos têm centenas de MB ou mais, ou quando há muitos lotes.


In [7]:
# grava hdfs_carga.sh | cria as zonas no HDFS e carrega os CSVs
gravar("hdfs_carga.sh", r'''#!/usr/bin/env bash
# HDFS | organiza as zonas do projeto no HDFS e carrega os CSVs.
# Requer Hadoop instalado e o HDFS no ar (start-dfs.sh). NÃO foi executado no desenvolvimento deste projeto.
#
# Uso:  bash hdfs_carga.sh [pasta_trusted_local] [pasta_raw_local] [pasta_quarentena_local]
#   padrão: dados_trusted   (na VM 3, só chegam os trusted; raw e quarentena ficam na VM 2)
set -euo pipefail

RAIZ=/gardenia
TRUSTED_LOCAL=${1:-dados_trusted}
RAW_LOCAL=${2:-}
QUAR_LOCAL=${3:-}

# 1) uma pasta por zona e por tabela (cada tabela em pasta própria, porque o Hive e o MapReduce leem pastas)
for zona in raw trusted quarentena; do
  for tabela in clientes produtos compras; do
    hdfs dfs -mkdir -p "$RAIZ/$zona/$tabela"
  done
done
hdfs dfs -mkdir -p "$RAIZ/mr" "$RAIZ/pig" "$RAIZ/hbase" "$RAIZ/warehouse"

# 2) carga: -put -f substitui o arquivo se ele já existir (reprocessamento de um lote)
for tabela in clientes produtos compras; do
  hdfs dfs -put -f "$TRUSTED_LOCAL"/trusted_${tabela}_*.csv "$RAIZ/trusted/$tabela/"
  [ -n "$RAW_LOCAL" ]  && hdfs dfs -put -f "$RAW_LOCAL"/${tabela}_*.csv "$RAIZ/raw/$tabela/" || true
  [ -n "$QUAR_LOCAL" ] && ls "$QUAR_LOCAL"/quarentena_${tabela}_*.csv >/dev/null 2>&1 \
                       && hdfs dfs -put -f "$QUAR_LOCAL"/quarentena_${tabela}_*.csv "$RAIZ/quarentena/$tabela/" || true
done

# 3) replicação: o padrão é 3 cópias de cada bloco. Em máquina única (pseudo-distribuído) só existe 1 DataNode: usar 1.
hdfs dfs -setrep -w 1 "$RAIZ" || true

# 4) conferência
hdfs dfs -ls -R "$RAIZ/trusted"
hdfs dfs -du -h "$RAIZ"
hdfs fsck "$RAIZ/trusted" -files -blocks | tail -15
''')

gravado: hadoop/hdfs_carga.sh  (36 linhas)


In [8]:
if tem("hdfs"):
    subprocess.run(["bash", "hadoop/hdfs_carga.sh", ENTRADA], check=True)
else:
    print("HDFS não instalado aqui. Para executar na VM 3:  bash hdfs_carga.sh dados_trusted")

HDFS não instalado aqui. Para executar na VM 3:  bash hdfs_carga.sh dados_trusted


## 4. MapReduce

**A tarefa:** receita líquida por categoria. O problema é que a categoria está na tabela de produtos e o valor da compra está na tabela de compras, então é preciso **juntar** as duas. Em MapReduce isso se faz com uma **junção no reduce** (*reduce-side join*), em 2 jobs:

**Job 1: junção, com uma linha de resumo por produto**
1. `map`: lê as linhas dos dois arquivos. Descobre a tabela pelo nome do arquivo e emite `lote:produto_id → P, categoria` ou `lote:produto_id → C, status, valor`.
2. `shuffle and sort` (feito pelo Hadoop): todas as linhas com a mesma chave vão para o mesmo reducer.
3. `reduce`: para cada produto, recebe a categoria e todas as compras juntas, soma e emite `categoria, receita, compras, canceladas`.

**Job 2: soma por categoria**
`map` repassa as linhas; `reduce` soma por categoria e calcula a % de cancelamento.

A chave é `lote:produto_id` porque os ids reiniciam a cada lote. Os valores circulam em **centavos inteiros**, para a soma não ter erro de arredondamento.

In [9]:
# grava mapper_join.py | Job 1, fase map
gravar("mapper_join.py", r'''#!/usr/bin/env python3
"""
MapReduce (Hadoop Streaming) | Job 1, fase MAP: junção de compras com produtos (reduce-side join).

Lê linhas CSV de trusted_produtos_N.csv e trusted_compras_N.csv. Como os dois arquivos entram no mesmo job, o mapper descobre
de qual tabela veio a linha pelo NOME DO ARQUIVO (o Hadoop informa em mapreduce_map_input_file) e marca o valor com P ou C.

Saída (separador TAB), com a chave "lote:produto_id":
    <lote>:<produto_id> TAB P TAB <categoria>
    <lote>:<produto_id> TAB C TAB <status> TAB <valor_liquido_em_centavos>
O reducer recebe todas as linhas de um produto juntas e junta categoria + compras.
"""
import csv
import os
import re
import sys


def contador(nome):
    # protocolo do Hadoop Streaming: aparece na tela do job, em "Counters"
    sys.stderr.write(f"reporter:counter:gardenia,{nome},1\n")


def centavos(texto):
    return int(round(float(texto) * 100))


def main():
    arquivo = os.environ.get("mapreduce_map_input_file") or os.environ.get("map_input_file") or ""
    m = re.search(r"trusted_(produtos|compras)_(\d+)\.csv", os.path.basename(arquivo))
    if not m:
        return                                   # arquivo que não faz parte do job
    tabela, lote = m.group(1), m.group(2)

    for linha in csv.reader(sys.stdin):
        if not linha or linha[0] == "id":        # cabeçalho (só existe no primeiro bloco do arquivo)
            continue
        try:
            if tabela == "produtos":             # id,nome,categoria,preco,...
                print(f"{lote}:{int(linha[0])}\tP\t{linha[2]}")
            else:                                # id,cliente_id,produto_id,quantidade,valor_total,desconto,data_compra,status,satisfacao
                liquido = centavos(linha[4]) - centavos(linha[5])
                print(f"{lote}:{int(linha[2])}\tC\t{linha[7]}\t{liquido}")
        except (ValueError, IndexError):
            contador("linhas_invalidas")


if __name__ == "__main__":
    main()
''')

gravado: hadoop/mapper_join.py  (49 linhas)


In [10]:
# grava reducer_join.py | Job 1, fase reduce
gravar("reducer_join.py", r'''#!/usr/bin/env python3
"""
MapReduce (Hadoop Streaming) | Job 1, fase REDUCE: para cada produto, soma as compras e anexa a categoria.

Entrada: linhas do mapper_join.py, ordenadas pela chave (o Hadoop faz essa ordenação, a fase "shuffle and sort").
Saída (TAB): categoria, receita_liquida_em_centavos (não canceladas), compras_nao_canceladas, compras_canceladas
"""
import sys
from itertools import groupby


def principal():
    linhas = (l.rstrip("\n").split("\t") for l in sys.stdin if l.strip())
    for chave, grupo in groupby(linhas, key=lambda campos: campos[0]):
        categoria, receita, validas, canceladas = None, 0, 0, 0
        for campos in grupo:
            if campos[1] == "P":
                categoria = campos[2]
            elif campos[2] == "cancelado":
                canceladas += 1
            else:
                receita += int(campos[3])
                validas += 1
        if validas + canceladas == 0:
            continue                              # produto sem compras
        print(f"{categoria or 'sem_produto'}\t{receita}\t{validas}\t{canceladas}")


if __name__ == "__main__":
    principal()
''')

gravado: hadoop/reducer_join.py  (30 linhas)


In [11]:
# grava mapper_soma.py | Job 2, fase map
gravar("mapper_soma.py", r'''#!/usr/bin/env python3
"""MapReduce (Hadoop Streaming) | Job 2, fase MAP: repassa as linhas (a chave é a categoria, o primeiro campo)."""
import sys

for linha in sys.stdin:
    if linha.strip():
        sys.stdout.write(linha)
''')

gravado: hadoop/mapper_soma.py  (7 linhas)


In [12]:
# grava reducer_soma.py | Job 2, fase reduce
gravar("reducer_soma.py", r'''#!/usr/bin/env python3
"""
MapReduce (Hadoop Streaming) | Job 2, fase REDUCE: soma os produtos de cada categoria.

Saída (TAB): categoria, receita_liquida (R$), compras_nao_canceladas, compras_canceladas, pct_cancelamento
"""
import sys
from itertools import groupby


def principal():
    linhas = (l.rstrip("\n").split("\t") for l in sys.stdin if l.strip())
    for categoria, grupo in groupby(linhas, key=lambda campos: campos[0]):
        receita = validas = canceladas = 0
        for _, r, v, c in grupo:
            receita += int(r); validas += int(v); canceladas += int(c)
        pct = 100 * canceladas / (validas + canceladas)
        print(f"{categoria}\t{receita / 100:.2f}\t{validas}\t{canceladas}\t{pct:.1f}")


if __name__ == "__main__":
    principal()
''')

gravado: hadoop/reducer_soma.py  (22 linhas)


In [13]:
# grava simular_streaming.py | simula o Hadoop Streaming na própria máquina
gravar("simular_streaming.py", r'''#!/usr/bin/env python3
"""
Simula o Hadoop Streaming na própria máquina, sem Hadoop instalado: map -> shuffle and sort -> reduce.

Serve para testar os 4 scripts (mapper/reducer) antes de rodar no cluster, e para mostrar o fluxo do MapReduce.
Faz o mesmo que o Hadoop faz:
  1. MAP: cada arquivo de entrada passa por um mapper (com a variável mapreduce_map_input_file definida);
  2. SHUFFLE AND SORT: as linhas de todos os mappers são ordenadas pela chave (campo antes do primeiro TAB);
  3. REDUCE: o reducer lê as linhas já agrupadas por chave.

Uso:  python3 simular_streaming.py --entrada dados_trusted --saida mr_local [--lotes 1,2]
"""
import argparse
import os
import re
import subprocess
import sys

AQUI = os.path.dirname(os.path.abspath(__file__))


def rodar(script, entrada_texto, env_extra=None):
    env = dict(os.environ, **(env_extra or {}))
    r = subprocess.run([sys.executable, os.path.join(AQUI, script)], input=entrada_texto, capture_output=True,
                       text=True, env=env, check=True)
    return r.stdout, r.stderr


def embaralhar_e_ordenar(texto):
    linhas = [l for l in texto.splitlines() if l]
    return "\n".join(sorted(linhas, key=lambda l: l.split("\t", 1)[0].encode("utf-8"))) + "\n"   # sort estável por chave


def job(mapper, reducer, arquivos, nome):
    saida_map, invalidas = [], 0
    for caminho in arquivos:
        with open(caminho, encoding="utf-8") as f:
            out, err = rodar(mapper, f.read(), {"mapreduce_map_input_file": os.path.abspath(caminho)})
        saida_map.append(out)
        invalidas += err.count("linhas_invalidas")
    mapeado = "".join(saida_map)
    ordenado = embaralhar_e_ordenar(mapeado)
    reduzido, _ = rodar(reducer, ordenado)
    print(f"[{nome}] map: {len(arquivos)} arquivo(s), {mapeado.count(chr(10))} linhas | "
          f"reduce: {len(set(l.split(chr(9), 1)[0] for l in ordenado.splitlines()))} chaves -> {reduzido.count(chr(10))} linhas"
          f" | linhas inválidas: {invalidas}")
    return reduzido


def main():
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--entrada", default="dados_trusted", help="pasta com trusted_produtos_N.csv e trusted_compras_N.csv")
    p.add_argument("--saida", default="mr_local", help="pasta dos resultados")
    p.add_argument("--lotes", help="lista de lotes separados por vírgula (padrão: todos)")
    a = p.parse_args()

    achados = {}
    for n in os.listdir(a.entrada):
        m = re.fullmatch(r"trusted_(produtos|compras)_(\d+)\.csv", n)
        if m:
            achados.setdefault(int(m.group(2)), set()).add(m.group(1))
    lotes = [n for n, t in sorted(achados.items()) if t == {"produtos", "compras"}]
    if a.lotes:
        lotes = [n for n in lotes if n in {int(x) for x in a.lotes.split(",")}]
    if not lotes:
        sys.exit(f"Nenhum lote completo (produtos + compras) em {a.entrada}")
    arquivos = [os.path.join(a.entrada, f"trusted_{t}_{n}.csv") for n in lotes for t in ("produtos", "compras")]

    os.makedirs(a.saida, exist_ok=True)
    por_produto = job("mapper_join.py", "reducer_join.py", arquivos, "job 1: junção")
    with open(os.path.join(a.saida, "mr_receita_por_produto.tsv"), "w", encoding="utf-8") as f:
        f.write(por_produto)
    # o job 2 lê a saída do job 1 (no Hadoop, ela fica no HDFS); aqui entra como um arquivo só
    tmp = os.path.join(a.saida, "_job1.tsv")
    with open(tmp, "w", encoding="utf-8") as f:
        f.write(por_produto)
    por_categoria = job("mapper_soma.py", "reducer_soma.py", [tmp], "job 2: soma por categoria")
    os.remove(tmp)
    destino = os.path.join(a.saida, "mr_receita_por_categoria.tsv")
    with open(destino, "w", encoding="utf-8") as f:
        f.write("categoria\treceita_liquida\tcompras\tcanceladas\tpct_cancelamento\n" + por_categoria)
    print(f"\nResultado: {destino}\n")
    print(open(destino, encoding="utf-8").read())


if __name__ == "__main__":
    main()
''')

gravado: hadoop/simular_streaming.py  (87 linhas)


### Execução (simulada) e conferência

In [14]:
r = subprocess.run([sys.executable, "hadoop/simular_streaming.py", "--entrada", ENTRADA, "--saida", "mr_local"],
                   capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else r.stderr)

[job 1: junção] map: 2 arquivo(s), 3489 linhas | reduce: 2305 chaves -> 926 linhas | linhas inválidas: 0
[job 2: soma por categoria] map: 1 arquivo(s), 926 linhas | reduce: 7 chaves -> 7 linhas | linhas inválidas: 0

Resultado: mr_local/mr_receita_por_categoria.tsv

categoria	receita_liquida	compras	canceladas	pct_cancelamento
Alimentos	266249.86	127	28	18.1
Brinquedos	236385.19	119	37	23.7
Casa	261623.94	121	37	23.4
Eletrônicos	244264.64	117	31	20.9
Esportes	324018.25	154	33	17.6
Livros	304085.78	148	35	19.1
Roupas	311430.64	152	45	22.8




In [15]:
# Conferência: o resultado do MapReduce precisa ser igual ao de um cálculo independente em pandas.
import pandas as pd

partes = []
for n in lotes:
    p = pd.read_csv(f"{ENTRADA}/trusted_produtos_{n}.csv")[["id", "categoria"]].rename(columns={"id": "produto_id"})
    c = pd.read_csv(f"{ENTRADA}/trusted_compras_{n}.csv")
    p["lote"] = c["lote"] = n
    partes.append(c.merge(p, on=["lote", "produto_id"]))
d = pd.concat(partes)
d["valor_liquido"] = d["valor_total"] - d["desconto"]
ok = d[d["status"] != "cancelado"]
esperado = ok.groupby("categoria").agg(receita=("valor_liquido", "sum"), compras=("id", "size"))
mr = pd.read_csv("mr_local/mr_receita_por_categoria.tsv", sep="\t").set_index("categoria")

assert (esperado["receita"] - mr["receita_liquida"]).abs().max() < 0.005, "receita diverge"
assert (esperado["compras"] == mr["compras"]).all(), "contagem diverge"
assert (d[d["status"] == "cancelado"].groupby("categoria").size() == mr["canceladas"]).all(), "cancelamentos divergem"
print("Conferido: MapReduce e pandas dão o mesmo resultado (receita, compras e cancelamentos por categoria).")

Conferido: MapReduce e pandas dão o mesmo resultado (receita, compras e cancelamentos por categoria).


In [16]:
# grava executar_mapreduce.sh | mesmos scripts, agora no cluster, via Hadoop Streaming e YARN
gravar("executar_mapreduce.sh", r'''#!/usr/bin/env bash
# MapReduce no cluster (Hadoop Streaming, executado pelo YARN). NÃO foi executado no desenvolvimento deste projeto;
# a lógica dos 4 scripts Python foi testada com simular_streaming.py.
#
# Pré-requisito: bash hdfs_carga.sh (os trusted precisam estar no HDFS).
# Uso:  bash executar_mapreduce.sh
set -euo pipefail

RAIZ=/gardenia
STREAMING_JAR=$(ls "$HADOOP_HOME"/share/hadoop/tools/lib/hadoop-streaming-*.jar | head -1)

hdfs dfs -rm -r -f "$RAIZ/mr/receita_por_produto" "$RAIZ/mr/receita_por_categoria"

# Job 1: junção compras x produtos. Lê as duas pastas; o mapper identifica a tabela pelo nome do arquivo.
hadoop jar "$STREAMING_JAR" \
  -D mapreduce.job.name=gardenia-join-compras-produtos \
  -D mapreduce.job.reduces=2 \
  -files mapper_join.py,reducer_join.py \
  -mapper "python3 mapper_join.py" \
  -reducer "python3 reducer_join.py" \
  -input "$RAIZ/trusted/produtos" -input "$RAIZ/trusted/compras" \
  -output "$RAIZ/mr/receita_por_produto"

# Job 2: soma por categoria (reducer único, o resultado é pequeno)
hadoop jar "$STREAMING_JAR" \
  -D mapreduce.job.name=gardenia-soma-por-categoria \
  -D mapreduce.job.reduces=1 \
  -files mapper_soma.py,reducer_soma.py \
  -mapper "python3 mapper_soma.py" \
  -reducer "python3 reducer_soma.py" \
  -input "$RAIZ/mr/receita_por_produto" \
  -output "$RAIZ/mr/receita_por_categoria"

hdfs dfs -cat "$RAIZ/mr/receita_por_categoria/part-*"
# acompanhar os jobs no YARN:  yarn application -list -appStates ALL
''')

gravado: hadoop/executar_mapreduce.sh  (35 linhas)


In [17]:
if tem("hadoop") and tem("hdfs"):
    subprocess.run(["bash", "hadoop/executar_mapreduce.sh"], cwd="hadoop", check=True)
else:
    print("Hadoop não instalado aqui. No cluster:  bash hdfs_carga.sh && bash executar_mapreduce.sh")

Hadoop não instalado aqui. No cluster:  bash hdfs_carga.sh && bash executar_mapreduce.sh


## 5. YARN

O **YARN** decide **onde e com quanto de CPU e memória** cada tarefa roda. Peças:

- **ResourceManager** (um por cluster): recebe os pedidos e reparte os recursos.
- **NodeManager** (um por máquina): executa as tarefas em **containers** (um pedaço de CPU e memória).
- **ApplicationMaster** (um por job): pede os containers e acompanha as tarefas.

No projeto, os dois jobs de MapReduce acima e o job Spark da VM 3 seriam aplicações YARN. O `pipeline_vm3.py` já aceita `--master yarn`: o código de processamento não muda, só o gerenciador.

**Limite do código atual:** o `pipeline_vm3.py` lê os CSVs de uma pasta local. Isso funciona numa máquina só (pseudo-distribuído). Com vários nós, a leitura teria de ser trocada para `hdfs://`.

In [18]:
# grava yarn_spark.sh | roda o Spark sob o YARN e inspeciona as aplicações
gravar("yarn_spark.sh", r'''#!/usr/bin/env bash
# YARN | rodar o job Spark da VM 3 sob o gerenciamento do YARN e inspecionar a fila de aplicações.
# NÃO foi executado no desenvolvimento deste projeto (precisa de Hadoop/YARN instalado).
#
# Limite do código atual: pipeline_vm3.py lê os CSVs de uma pasta LOCAL. Isso funciona em máquina única
# (pseudo-distribuído), onde os executores do YARN rodam na mesma máquina. Em cluster com vários nós, os dados teriam
# de ser lidos do HDFS (hdfs://...), o que exige trocar a leitura em lotes_trusted/ler_tabela.
set -euo pipefail

export HADOOP_CONF_DIR=${HADOOP_CONF_DIR:-$HADOOP_HOME/etc/hadoop}    # o Spark descobre o ResourceManager por aqui

# 1) o mesmo script da VM 3, trocando só o master (local[*] -> yarn)
python3 pipeline_vm3.py processar --master yarn --forcar --memoria 2g

# 2) equivalente com spark-submit, escolhendo recursos de cada executor
# spark-submit --master yarn --deploy-mode client --num-executors 2 --executor-memory 1g --executor-cores 1 pipeline_vm3.py processar --master yarn

# 3) acompanhamento
yarn node -list                      # nós do cluster e recursos
yarn application -list -appStates ALL
# yarn application -status <application_id>
# yarn logs -applicationId <application_id>
# yarn application -kill <application_id>
''')

gravado: hadoop/yarn_spark.sh  (23 linhas)


## 6. Hive

O Hive deixa você consultar os arquivos do HDFS com **SQL**. Ele guarda só o catálogo (nomes e colunas das tabelas); os dados continuam nos arquivos. Cada consulta é compilada para jobs distribuídos.

O script cria **tabelas externas** sobre `/gardenia/trusted/...` (apagar a tabela não apaga o arquivo), **views** que convertem os tipos e extraem o lote do nome do arquivo, e as mesmas 4 consultas do notebook 7.5. A view de clientes **não expõe nome, e-mail, telefone nem CPF**, para respeitar a LGPD.

In [19]:
# grava hive_gardenia.hql | tabelas externas, views e as 4 consultas
gravar("hive_gardenia.hql", r'''-- Hive | tabelas externas sobre os CSVs do HDFS e as mesmas consultas do notebook 7.5 (Spark SQL), agora em HiveQL.
-- NÃO foi executado no desenvolvimento deste projeto (precisa de Hive instalado). Uso:  hive -f hive_gardenia.hql
--
-- EXTERNAL: o Hive guarda só os metadados; os arquivos continuam no HDFS (DROP TABLE não apaga os dados).
-- OpenCSVSerde lê tudo como STRING, por isso as views fazem o CAST.

CREATE DATABASE IF NOT EXISTS gardenia;
USE gardenia;

CREATE EXTERNAL TABLE IF NOT EXISTS clientes (
  id STRING, nome STRING, email STRING, idade STRING, telefone STRING, cpf STRING,
  data_nascimento STRING, data_cadastro STRING, cadastro_incompleto STRING)
ROW FORMAT SERDE 'org.apache.hadoop.hive.serde2.OpenCSVSerde'
WITH SERDEPROPERTIES ("separatorChar" = ",", "quoteChar" = "\"")
STORED AS TEXTFILE LOCATION '/gardenia/trusted/clientes'
TBLPROPERTIES ("skip.header.line.count" = "1");

CREATE EXTERNAL TABLE IF NOT EXISTS produtos (
  id STRING, nome STRING, categoria STRING, preco STRING, estoque STRING, peso STRING,
  satisfacao_media STRING, data_cadastro STRING)
ROW FORMAT SERDE 'org.apache.hadoop.hive.serde2.OpenCSVSerde'
WITH SERDEPROPERTIES ("separatorChar" = ",", "quoteChar" = "\"")
STORED AS TEXTFILE LOCATION '/gardenia/trusted/produtos'
TBLPROPERTIES ("skip.header.line.count" = "1");

CREATE EXTERNAL TABLE IF NOT EXISTS compras (
  id STRING, cliente_id STRING, produto_id STRING, quantidade STRING, valor_total STRING, desconto STRING,
  data_compra STRING, status STRING, satisfacao STRING)
ROW FORMAT SERDE 'org.apache.hadoop.hive.serde2.OpenCSVSerde'
WITH SERDEPROPERTIES ("separatorChar" = ",", "quoteChar" = "\"")
STORED AS TEXTFILE LOCATION '/gardenia/trusted/compras'
TBLPROPERTIES ("skip.header.line.count" = "1");

-- O número do lote está no nome do arquivo (trusted_compras_2.csv): INPUT__FILE__NAME é a coluna virtual do Hive.
-- Os ids reiniciam a cada lote, então a chave de junção é (lote, id), como no DW e no Spark.
CREATE VIEW IF NOT EXISTS vw_produtos AS
SELECT regexp_extract(INPUT__FILE__NAME, '_([0-9]+)[.]csv$', 1) AS lote,
       CAST(id AS INT) AS produto_id, nome AS produto, categoria, CAST(preco AS DOUBLE) AS preco
FROM produtos;

-- clientes sem os dados pessoais (nome, e-mail, telefone, CPF): é a view que as análises devem usar (LGPD)
CREATE VIEW IF NOT EXISTS vw_clientes AS
SELECT regexp_extract(INPUT__FILE__NAME, '_([0-9]+)[.]csv$', 1) AS lote,
       CAST(id AS INT) AS cliente_id, CAST(idade AS INT) AS idade,
       CASE WHEN idade IS NULL OR idade = '' THEN 'sem idade'
            WHEN CAST(idade AS INT) < 25 THEN '18-24' WHEN CAST(idade AS INT) < 35 THEN '25-34'
            WHEN CAST(idade AS INT) < 45 THEN '35-44' WHEN CAST(idade AS INT) < 55 THEN '45-54'
            WHEN CAST(idade AS INT) < 65 THEN '55-64' ELSE '65+' END AS faixa_etaria
FROM clientes;

CREATE VIEW IF NOT EXISTS vw_compras AS
SELECT regexp_extract(INPUT__FILE__NAME, '_([0-9]+)[.]csv$', 1) AS lote,
       CAST(id AS INT) AS compra_id, CAST(cliente_id AS INT) AS cliente_id, CAST(produto_id AS INT) AS produto_id,
       CAST(quantidade AS INT) AS quantidade, CAST(valor_total AS DOUBLE) AS valor_total, CAST(desconto AS DOUBLE) AS desconto,
       CAST(data_compra AS TIMESTAMP) AS data_compra, status, CAST(satisfacao AS INT) AS satisfacao
FROM compras;

CREATE VIEW IF NOT EXISTS fato_vendas AS
SELECT c.lote, c.compra_id, c.cliente_id, c.produto_id, p.produto, p.categoria, cl.faixa_etaria,
       c.valor_total - c.desconto AS valor_liquido,
       date_format(c.data_compra, 'yyyy-MM') AS mes,
       concat(year(c.data_compra), '-T', quarter(c.data_compra)) AS trimestre,
       c.status, (c.status <> 'cancelado') AS valida, c.satisfacao
FROM vw_compras c
JOIN vw_produtos p  ON c.lote = p.lote  AND c.produto_id = p.produto_id
JOIN vw_clientes cl ON c.lote = cl.lote AND c.cliente_id = cl.cliente_id;

-- ---------------------------------------------------------------- consultas (as mesmas do notebook 7.5)
-- 1) receita por trimestre e categoria, com a participação no trimestre (função de janela)
SELECT trimestre, categoria, ROUND(SUM(valor_liquido), 2) AS receita_liquida,
       ROUND(100 * SUM(valor_liquido) / SUM(SUM(valor_liquido)) OVER (PARTITION BY trimestre), 1) AS pct_do_trimestre
FROM fato_vendas WHERE valida
GROUP BY trimestre, categoria
ORDER BY trimestre, receita_liquida DESC;

-- 2) receita mensal e variação sobre o mês anterior (LAG)
SELECT mes, ROUND(receita, 2) AS receita_liquida,
       ROUND(100 * (receita - LAG(receita) OVER (ORDER BY mes)) / LAG(receita) OVER (ORDER BY mes), 1) AS variacao_pct
FROM (SELECT mes, SUM(valor_liquido) AS receita FROM fato_vendas WHERE valida GROUP BY mes) m
ORDER BY mes;

-- 3) os 3 produtos de maior receita em cada categoria
SELECT * FROM (
  SELECT categoria, lote, produto_id, produto, ROUND(SUM(valor_liquido), 2) AS receita_liquida,
         ROW_NUMBER() OVER (PARTITION BY categoria ORDER BY SUM(valor_liquido) DESC) AS posicao
  FROM fato_vendas WHERE valida
  GROUP BY categoria, lote, produto_id, produto) t
WHERE posicao <= 3
ORDER BY categoria, posicao;

-- 4) cancelamento por faixa etária
SELECT faixa_etaria, COUNT(*) AS compras, SUM(CASE WHEN valida THEN 0 ELSE 1 END) AS canceladas,
       ROUND(100 * AVG(CASE WHEN valida THEN 0.0 ELSE 1.0 END), 1) AS pct_cancelamento
FROM fato_vendas
GROUP BY faixa_etaria
HAVING COUNT(*) >= 100
ORDER BY pct_cancelamento DESC;
''')

gravado: hadoop/hive_gardenia.hql  (97 linhas)


## 7. Pig

O Pig descreve o processamento como um **fluxo de dados**: carregar, filtrar, juntar, agrupar, gravar. O Pig transforma isso em jobs MapReduce, e por isso o script abaixo, bem mais curto, faz o papel dos 4 arquivos Python do MapReduce. O resultado esperado é o mesmo, mas o script **não foi executado**: confira com a saída do MapReduce.

In [20]:
# grava pig_gardenia.pig | receita líquida por categoria em Pig Latin
gravar("pig_gardenia.pig", r'''-- Pig | o mesmo cálculo do MapReduce (receita líquida por categoria), escrito como fluxo de dados em Pig Latin.
-- NÃO foi executado no desenvolvimento deste projeto (precisa de Pig e do piggybank instalados).
--
-- Uso:  pig -param LOTE=1 pig_gardenia.pig
-- Cada comando vira um ou mais jobs MapReduce; o Pig monta o plano, no lugar de escrevermos mapper e reducer.
%default LOTE '1'
%default RAIZ '/gardenia'

REGISTER /usr/lib/pig/piggybank.jar;     -- o caminho do piggybank muda conforme a instalação
-- CSVExcelStorage entende campos entre aspas e ignora o cabeçalho
DEFINE CSV org.apache.pig.piggybank.storage.CSVExcelStorage(',', 'NO_MULTILINE', 'UNIX', 'SKIP_INPUT_HEADER');

produtos = LOAD '$RAIZ/trusted/produtos/trusted_produtos_$LOTE.csv' USING CSV
           AS (id:int, nome:chararray, categoria:chararray, preco:double, estoque:int, peso:double,
               satisfacao_media:double, data_cadastro:chararray);
compras  = LOAD '$RAIZ/trusted/compras/trusted_compras_$LOTE.csv' USING CSV
           AS (id:int, cliente_id:int, produto_id:int, quantidade:int, valor_total:double, desconto:double,
               data_compra:chararray, status:chararray, satisfacao:int);

-- 1) receita líquida por categoria (compras não canceladas)
validas = FILTER compras BY status != 'cancelado';
juncao  = JOIN validas BY produto_id, produtos BY id;
base    = FOREACH juncao GENERATE produtos::categoria AS categoria,
                                  (validas::valor_total - validas::desconto) AS valor_liquido;
por_cat = GROUP base BY categoria;
receita = FOREACH por_cat GENERATE group AS categoria, COUNT(base) AS compras,
                                   ROUND_TO(SUM(base.valor_liquido), 2) AS receita_liquida;
receita_ord = ORDER receita BY receita_liquida DESC;
STORE receita_ord INTO '$RAIZ/pig/receita_por_categoria_$LOTE' USING PigStorage(',');

-- 2) compras por status
por_status = GROUP compras BY status;
contagem   = FOREACH por_status GENERATE group AS status, COUNT(compras) AS compras;
STORE contagem INTO '$RAIZ/pig/compras_por_status_$LOTE' USING PigStorage(',');
''')

gravado: hadoop/pig_gardenia.pig  (34 linhas)


## 8. Sqoop

Sem banco relacional de origem, o Sqoop não é usado no projeto. O script mostra como seria a importação, a importação incremental e a exportação de um resultado de volta ao banco, **caso as compras ficassem num MySQL**. O Apache Sqoop foi desativado; hoje o caminho recomendado é `spark.read.jdbc`.

In [21]:
# grava sqoop_exemplo.sh | exemplo, não executável neste projeto
gravar("sqoop_exemplo.sh", r'''#!/usr/bin/env bash
# Sqoop | transferência entre banco relacional e HDFS.
# EXEMPLO, NÃO executável neste projeto: aqui não existe banco relacional de origem (a fonte são arquivos CSV
# do simulador, copiados por SCP). O Sqoop passaria a fazer sentido se as compras ficassem num MySQL/PostgreSQL.
# Atenção: o Apache Sqoop foi desativado pela Apache; o equivalente atual é o Spark lendo por JDBC (spark.read.jdbc).
set -euo pipefail

JDBC="jdbc:mysql://IP_DO_BANCO:3306/gardenia"      # banco hipotético
USUARIO=gardenia_leitura                            # senha: use --password-file, nunca a senha na linha de comando

# importar a tabela compras para a zona raw do HDFS (4 mappers lendo faixas de id em paralelo)
sqoop import --connect "$JDBC" --username "$USUARIO" --password-file /user/gardenia/.senha \
  --table compras --split-by id --num-mappers 4 \
  --target-dir /gardenia/raw/compras --as-textfile --fields-terminated-by ','

# importação incremental: só o que chegou depois da última carga
sqoop import --connect "$JDBC" --username "$USUARIO" --password-file /user/gardenia/.senha \
  --table compras --target-dir /gardenia/raw/compras \
  --incremental append --check-column id --last-value 5000

# o caminho inverso: devolver um resultado do HDFS para uma tabela do banco (por exemplo, os segmentos de clientes)
sqoop export --connect "$JDBC" --username "$USUARIO" --password-file /user/gardenia/.senha \
  --table segmentos_clientes --export-dir /gardenia/ml/segmentos --input-fields-terminated-by ','
''')

gravado: hadoop/sqoop_exemplo.sh  (23 linhas)


## 9. HBase

O HBase guarda linhas **ordenadas pela chave**, e a leitura por chave (ou por prefixo da chave) é muito rápida. O desenho da chave é a decisão principal:

`<lote>#<cliente_id>#<data_compra>#<compra_id>`, com zeros à esquerda para manter a ordem.

Com o cliente no começo da chave, todas as compras dele ficam juntas e saem em ordem de data numa só leitura. Uma tabela com a chave `compra_id` não responderia a essa pergunta sem varrer tudo.

In [22]:
# grava preparar_hbase.py | gera o TSV com a chave de linha
gravar("preparar_hbase.py", r'''#!/usr/bin/env python3
"""
HBase | prepara o arquivo TSV das compras com a chave de linha (row key) pensada para consulta por cliente.

Chave:  <lote 2 dígitos>#<cliente_id 7 dígitos>#<data_compra AAAAMMDDhhmmss>#<compra_id 7 dígitos>
Por quê: o HBase guarda as linhas ordenadas pela chave. Com o cliente no início, todas as compras de um cliente ficam
         juntas e saem de uma só leitura por prefixo (scan), em ordem de data. Os dígitos fixos (zeros à esquerda) mantêm
         a ordem correta ("0000042" vem antes de "0000100"; sem os zeros, "100" viria antes de "42").

Uso:  python3 preparar_hbase.py --entrada dados_trusted --saida hbase_compras.tsv
Colunas do TSV (família "d"):  d:produto_id d:quantidade d:valor_liquido d:status d:satisfacao
"""
import argparse
import csv
import os
import re
import sys
from datetime import datetime


def centavos(texto):
    return int(round(float(texto) * 100))


def linhas_compras(pasta):
    for nome in sorted(os.listdir(pasta)):
        m = re.fullmatch(r"trusted_compras_(\d+)\.csv", nome)
        if not m:
            continue
        lote = int(m.group(1))
        with open(os.path.join(pasta, nome), encoding="utf-8", newline="") as f:
            for r in csv.DictReader(f):
                data = datetime.fromisoformat(r["data_compra"])
                chave = f"{lote:02d}#{int(r['cliente_id']):07d}#{data:%Y%m%d%H%M%S}#{int(r['id']):07d}"
                liquido = (centavos(r["valor_total"]) - centavos(r["desconto"])) / 100
                yield [chave, r["produto_id"], r["quantidade"], f"{liquido:.2f}", r["status"], r["satisfacao"]]


def main():
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--entrada", default="dados_trusted")
    p.add_argument("--saida", default="hbase_compras.tsv")
    a = p.parse_args()
    n = 0
    with open(a.saida, "w", encoding="utf-8", newline="") as f:
        for linha in linhas_compras(a.entrada):
            f.write("\t".join(linha) + "\n")
            n += 1
    if n == 0:
        sys.exit(f"Nenhum trusted_compras_N.csv em {a.entrada}")
    print(f"{n} linhas gravadas em {a.saida}")


if __name__ == "__main__":
    main()
''')

gravado: hadoop/preparar_hbase.py  (55 linhas)


In [23]:
# grava hbase_gardenia.sh | cria a tabela, carrega e consulta
gravar("hbase_gardenia.sh", r'''#!/usr/bin/env bash
# HBase | tabela de compras com acesso rápido ao histórico de um cliente.
# NÃO foi executado no desenvolvimento deste projeto (precisa de HBase instalado). O preparar_hbase.py foi testado.
#
# Quando o HBase faz sentido aqui: "mostre as últimas compras do cliente X" (leitura pontual, em milissegundos).
# Quando NÃO faz: somar a receita do ano (agregação sobre tudo). Isso é trabalho para Spark/Hive.
set -euo pipefail

python3 preparar_hbase.py --entrada dados_trusted --saida hbase_compras.tsv
hdfs dfs -mkdir -p /gardenia/hbase
hdfs dfs -put -f hbase_compras.tsv /gardenia/hbase/compras.tsv

hbase shell <<'SHELL'
create_namespace 'gardenia'
create 'gardenia:compras', {NAME => 'd', VERSIONS => 1}
SHELL

# carga em massa a partir do TSV (é um job MapReduce)
hbase org.apache.hadoop.hbase.mapreduce.ImportTsv \
  -Dimporttsv.columns=HBASE_ROW_KEY,d:produto_id,d:quantidade,d:valor_liquido,d:status,d:satisfacao \
  gardenia:compras /gardenia/hbase/compras.tsv

hbase shell <<'SHELL'
count 'gardenia:compras'
# todas as compras do cliente 42 no lote 01, em ordem de data (scan por prefixo da chave)
scan 'gardenia:compras', {STARTROW => '01#0000042#', STOPROW => '01#0000042$'}
# uma compra específica
get 'gardenia:compras', '01#0000042#20260505185835#0000001'
SHELL
''')

gravado: hadoop/hbase_gardenia.sh  (29 linhas)


In [24]:
r = subprocess.run([sys.executable, "hadoop/preparar_hbase.py", "--entrada", ENTRADA, "--saida", "hbase_compras.tsv"],
                   capture_output=True, text=True)
print(r.stdout or r.stderr)

# Simula a leitura por prefixo do HBase: linhas ordenadas pela chave, e o "scan" pega o intervalo do cliente.
linhas = sorted(l.rstrip("\n").split("\t") for l in open("hbase_compras.tsv", encoding="utf-8"))
cliente_exemplo = max({l[0].split("#")[1] for l in linhas},
                      key=lambda c: sum(1 for l in linhas if l[0].split("#")[1] == c))
lote0 = f"{lotes[0]:02d}"
inicio, fim = f"{lote0}#{cliente_exemplo}#", f"{lote0}#{cliente_exemplo}$"
resultado = [l for l in linhas if inicio <= l[0] < fim]
print(f"\nscan {inicio!r} até {fim!r}: {len(resultado)} compra(s) do cliente {int(cliente_exemplo)} no lote {lote0}, em ordem de data")
print(pd.DataFrame(resultado, columns=["row key", "d:produto_id", "d:quantidade", "d:valor_liquido", "d:status", "d:satisfacao"]).to_string(index=False))

1184 linhas gravadas em hbase_compras.tsv


scan '01#0001847#' até '01#0001847$': 3 compra(s) do cliente 1847 no lote 01, em ordem de data
                          row key d:produto_id d:quantidade d:valor_liquido  d:status d:satisfacao
01#0001847#20251110173144#0002261         4938            1         2949.91 cancelado            3
01#0001847#20260506173144#0002252         4892            6         3066.02 cancelado             
01#0001847#20260925173144#0002144         3686            5         3371.05   enviado            1


## 10. Resumo e limitações

| Ferramenta | Situação neste projeto |
|---|---|
| MapReduce | **Executado** em simulação local do Hadoop Streaming e **conferido com pandas**. Não executado em cluster. |
| HBase | **Executado** só a preparação da chave e a leitura por prefixo (simulada). Tabela e carga não executadas. |
| HDFS, YARN, Hive, Pig, Sqoop | **Roteiro** (scripts prontos, não executados). O Sqoop não se aplica ao projeto atual. |
| Spark | **Executado** nos notebooks 07 e 7.5 e no `pipeline_vm3.py`, que cobrem as mesmas análises. |

**Limitações**

- Os scripts de cluster **não foram testados contra um Hadoop real**. Podem precisar de ajustes de caminho (`HADOOP_HOME`, `piggybank.jar`) e de versão.
- Com poucos MB por lote, o Hadoop **não dá ganho de desempenho**. O que se mostra é a arquitetura, e ela só se paga com volume maior.
- O Hadoop Streaming usa Python e é mais lento que um MapReduce em Java; foi escolhido por ser legível e testável sem instalar nada.
- O `pipeline_vm3.py` com `--master yarn` só funciona em máquina única, enquanto lê de pasta local.

**Para entregar:** os arquivos da pasta `hadoop/` e este notebook. Se o professor pedir evidência de execução no cluster, instale o Hadoop na VM 3 em modo pseudo-distribuído e rode `hdfs_carga.sh` e `executar_mapreduce.sh`; a saída desses dois scripts é a evidência.